# IoT Hub Monitor — обучение LSTM-форкастера `lstm_lf_resid` в Google Colab (Фаза 4)

Пайплайн: **БД → CSV → Colab GPU (обучение) → веса + ONNX → мак/Render (инференс)**.

Этот ноутбук **импортирует те же `ml/`-модули, что и прод** (через `git clone` публичной
репы) — он НЕ дублирует логику моделей и persistence. Обученный артефакт
(`<key>.manifest.json` + `.pt`/`.npz`/`_hw.joblib` + `.onnx`) сохраняется тем же
`ModelPersistenceMixin.save` + `export_onnx`, что и локально.

**Что нужно заранее:** по одному CSV на каждое из 4 temperature-устройств, выгруженных
командой `python manage.py export_training_data --device TEMP-001 --metric temperature --output train_TEMP-001.csv`.

**Про GPU и ONNX.** Учим на **GPU (T4)** — для LSTM это ~5× быстрее CPU. GPU и CPU дают
чуть разные веса (другой порядок float-операций), но это НЕ проблема: **боевой инференс
идёт через ONNX** (метод `lstm_lf_resid_onnx`, `onnxruntime`, без torch) — ONNX даёт
одинаковый вывод на любом железе. Ноутбук кроме `.pt` пишет `.onnx` (`export_onnx` после
`save`). Боевой `ml/`-код остаётся CPU-чистым; GPU включается ТОЛЬКО здесь патчем (2b).

---

**Маршрут запуска:** `1 → 2 → 2b → 4 → 6`. Ячейка 2 ставит statsmodels/onnx и
**перезапускает рантайм** (это норма, не ошибка) — после рестарта начните заново с
ячейки 1. Перед обучением: Runtime → T4 GPU.

## 1. Клонируем репу и добавляем её корень в `sys.path`

`iot_hub/__init__.py` отсутствует намеренно — пакет резолвится как implicit namespace
package (Python 3), ровно как в `tests/`. Достаточно положить корень репы в `sys.path`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/vadim-white/IoT-hub-monitor.git"
REPO_DIR = "IoT-hub-monitor"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL

repo_root = os.path.abspath(REPO_DIR)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("repo_root:", repo_root)

## 2. Фиксируем версии зависимостей

Версии **из `requirements.txt`** — критично для воспроизводимости весов. Colab несёт свои
torch/numpy; без пина веса могут чуть отличаться. Django НЕ ставим — `ml/`-ядро от него
не зависит (`loader.py` с ORM в Colab не используем, ряд собираем из CSV).

In [ ]:
# Colab уже несёт рабочие torch+CUDA и numpy — их НЕ трогаем (жёсткие пины ломают ABI:
# "numpy.dtype size changed"). Ставим statsmodels (внутренний HW для lstm_lf_resid),
# sklearn, и onnx+onnxruntime (экспорт .pt→.onnx, инкр.5) — pip подберёт версии под numpy.
import importlib.util
_need = [p for p in ("statsmodels", "onnx", "onnxruntime")
         if importlib.util.find_spec(p) is None]
if _need:
    !pip install -q statsmodels scikit-learn onnx onnxruntime
    print("\n" + "="*60)
    print("пакеты установлены. ТЕПЕРЬ ПЕРЕЗАПУСТИТЕ РАНТАЙМ:")
    print("  Runtime → Restart session (или Ctrl+M .)")
    print("После рестарта прогоните ячейки заново: 1 → 2 → 2b → 4 → 6")
    print("(ячейка 2 во второй раз пройдёт мгновенно — пакеты уже стоят)")
    print("="*60)
else:
    # проверяем, что numpy-ABI цел (импорт не падает) — после рестарта это пройдёт
    import statsmodels.api as sm
    import onnxruntime  # noqa: F401
    print("statsmodels + onnxruntime OK, ABI цел — можно продолжать к 2b")

## 2b. GPU-патч для обучения lstm_lf_resid (только в этом ноутбуке)

Боевой класс `LSTMLevelFixResidualForecaster` считает на CPU (тензоры без `.to(device)`),
чтобы прод оставался CPU-чистым. Здесь, **не трогая репозиторий**, переопределяем методы
так, чтобы обучение шло на GPU, а сохранение/экспорт — на CPU:

- `_build_and_train` — сеть и обучающие тензоры на `cuda`;
- `forecast` — вход на `cuda`, выход через `.cpu().numpy()`;
- `_dump_state` — веса сохраняются как **CPU**-state_dict, чтобы `.pt` грузился на
  маке/Render без GPU (иначе `torch.load` искал бы cuda-устройство);
- `export_onnx` — модель на CPU на время `torch.onnx.export` (трассировка по CPU-dummy).

Логика модели (level-fix на остатке HW) идентична боевой — меняется только устройство.

In [ ]:
# guard: гарантируем iot_hub в sys.path (после рестарта рантайма ячейка 1 могла не
# выполниться повторно). Идемпотентно — если путь уже есть, ничего не делает.
import sys, os
_root = os.path.abspath("IoT-hub-monitor")
if not os.path.isdir(_root):
    !git clone --depth 1 https://github.com/vadim-white/IoT-hub-monitor.git
if _root not in sys.path:
    sys.path.insert(0, _root)

# GPU-патч: переопределяем методы lstm_lf_resid на cuda (репозиторий не меняем)
import numpy as np, torch
from iot_hub.apps.telemetry.ml.forecasters.lstm_level_fix_residual import (
    LSTMLevelFixResidualForecaster as LF)
from iot_hub.apps.telemetry.ml.forecast_base import ForecastResult, future_timestamps
from iot_hub.apps.telemetry.ml.torch_utils import build_lstm_seq2seq, set_torch_determinism

DEVICE_T = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("обучение на:", DEVICE_T)

def _build_and_train_gpu(self, horizon):
    set_torch_determinism(self.random_state)
    r, w, std = self._resid, self.window, self._std
    n = len(r)
    X, D = [], []
    for t in range(w, n - horizon + 1):
        level = r[t - 1]
        X.append((r[t - w:t] - level) / std)
        D.append((r[t:t + horizon] - level) / std)
    if not X:
        self._model = None
        return
    X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1).to(DEVICE_T)
    D = torch.tensor(np.array(D), dtype=torch.float32).to(DEVICE_T)
    model = build_lstm_seq2seq(self.hidden, horizon).to(DEVICE_T)
    opt = torch.optim.Adam(model.parameters(), lr=self.lr)
    loss_fn = torch.nn.MSELoss()
    model.train()
    for _ in range(self.epochs):
        opt.zero_grad(); loss = loss_fn(model(X), D); loss.backward(); opt.step()
    model.eval()
    self._model = model

def _forecast_gpu(self, horizon):
    if self._model is None or self._horizon != horizon:
        self._horizon = horizon
        self._build_and_train(horizon)
    hw_fc = np.asarray(self._hw.forecast(horizon).mean, dtype=float)
    if self._model is None:
        mean = hw_fc
    else:
        x = torch.tensor(self._tail, dtype=torch.float32).reshape(1, self.window, 1).to(DEVICE_T)
        with torch.no_grad():
            d = self._model(x).cpu().numpy().ravel()   # cuda → cpu перед numpy
        resid_fc = self._r_level + d * self._std
        mean = hw_fc + resid_fc[:horizon]
    ts = future_timestamps(self._last_ts, self._step, horizon)
    return ForecastResult(timestamps=ts, mean=mean, lower=None, upper=None, horizon=horizon,
        meta={"method": self.name, "seasonal_periods": self.seasonal_periods,
              "params": {"window": self.window, "epochs": self.epochs,
                         "hidden": self.hidden, "random_state": self.random_state}})

_orig_dump = LF._dump_state
def _dump_state_cpu(self, stem):
    # веса на cuda → перенести модель на cpu перед сохранением, чтобы .pt грузился
    # на маке/Render без GPU; затем вернуть на DEVICE_T (вдруг ещё нужен forecast)
    if self._model is not None:
        self._model.to("cpu")
    _orig_dump(self, stem)
    if self._model is not None:
        self._model.to(DEVICE_T)

_orig_export = LF.export_onnx
def _export_onnx_cpu(self, stem, opset: int = 17):
    # torch.onnx.export трассирует на CPU-dummy → сеть и веса должны быть на CPU;
    # переносим на cpu на время экспорта, потом возвращаем на DEVICE_T (инкр.5)
    if self._model is not None:
        self._model.to("cpu")
    try:
        return _orig_export(self, stem, opset=opset)
    finally:
        if self._model is not None:
            self._model.to(DEVICE_T)

LF._build_and_train = _build_and_train_gpu
LF.forecast = _forecast_gpu
LF._dump_state = _dump_state_cpu
LF.export_onnx = _export_onnx_cpu
print("патч применён: lstm_lf_resid обучается на", DEVICE_T, "(save/export_onnx → CPU)")

## 4. Обучить все 4 temperature-устройства за один прогон

Цикл по TEMP-001/004/006/008 — те же, что прогоняются на Render. Для каждого:
загрузить свой CSV, `fit` → `forecast(HORIZON)` (строит и обучает сеть в Colab!) →
`save` → `export_onnx` → `evaluate_and_select` (оценка готовых весов HW vs LSTM на
hold-out, инкр.6-7, без переобучения — пишет selection-файл рядом с весами). Гиперпараметры фиксированы в `LSTM_LF_RESID_DEFAULTS` — не трогаем.

**Перед запуском:** выполните GPU-патч (ячейка 2b), затем загрузите 4 файла
`train_TEMP-001.csv … train_TEMP-008.csv` (виджет ниже) или смонтируйте Drive.
Имена должны совпадать с шаблоном `CSV_TMPL`.

In [ ]:
# guard: гарантируем iot_hub в sys.path (после рестарта рантайма ячейка 1 могла не
# выполниться повторно). Идемпотентно — если путь уже есть, ничего не делает.
import sys, os
_root = os.path.abspath("IoT-hub-monitor")
if not os.path.isdir(_root):
    !git clone --depth 1 https://github.com/vadim-white/IoT-hub-monitor.git
if _root not in sys.path:
    sys.path.insert(0, _root)

from google.colab import files
from iot_hub.apps.telemetry.ml.dataset_csv import load_series_from_csv
from iot_hub.apps.telemetry.ml.forecasters import build_forecaster
from iot_hub.apps.telemetry.ml.cli_params import forecaster_params
from iot_hub.apps.telemetry.ml.persistence import model_key
from iot_hub.apps.telemetry.ml.forecaster_select import evaluate_and_select, save_selection, selection_key
import time

DEVICES = ["TEMP-001", "TEMP-004", "TEMP-006", "TEMP-008"]
METRIC  = "temperature"
HORIZON = 36                   # зашивается в ONNX-граф; совпадает с forecast_telemetry
CSV_TMPL = "train_{dev}.csv"   # шаблон имени выгруженного CSV (export_training_data)

uploaded = files.upload()      # выберите все 4 файла train_TEMP-XXX.csv сразу
print("загружено:", list(uploaded))

for dev in DEVICES:
    csv = CSV_TMPL.format(dev=dev)
    if csv not in uploaded:
        print(f"  {dev}: пропуск — нет {csv}"); continue
    series = load_series_from_csv(csv, dev, METRIC)
    model = build_forecaster("lstm_lf_resid", **forecaster_params("lstm_lf_resid", {}))
    t0 = time.perf_counter()
    model.fit(series)
    model.forecast(HORIZON)    # ВАЖНО: строит и обучает LSTM-сеть здесь, в Colab
    stem = model_key(model.name, dev, METRIC)
    model.save(stem)           # .npz + .pt + _hw.joblib + манифесты
    model.export_onnx(stem)    # .onnx — боевой инференс без torch (инкр.5)
    # авто-выбор (инкр.6): мини-бэктест HW vs lstm_lf_resid на этом ряде (torch есть) →
    # selection-файл рядом с весами. Render потом ЧИТАЕТ его без torch.
    sel = evaluate_and_select(series, device_sn=dev, metric_type=METRIC, horizon=HORIZON)  # hold-out, без переобучения LSTM
    sel_path = save_selection(selection_key(dev, METRIC), sel)
    print(f"  {dev}: обучен за {time.perf_counter()-t0:.1f}с → {stem.name} "
          f"(+ .onnx, auto={sel['best_method']} mae_hw={sel['mae_hw']} mae_lstm={sel['mae_lstm']})")

print("\nГотово. Скачайте zip последней ячейкой (## 6) — .onnx уже внутри models/.")

## 6. Скачиваем веса (с .onnx внутри)

Архив содержит `.pt`/`.npz`/`_hw.joblib` + `.onnx` + `_forecaster_select.json` + манифесты. Распакуйте локально в
`iot_hub/apps/telemetry/ml/models/`, затем проверьте боевой ONNX-инференс (без torch):

```bash
python manage.py forecast_telemetry --device TEMP-001 --metric temperature \
    --method auto --use-cache --write-alerts
```

В логе должно появиться `[cache] загружено` — взяты готовые веса, без переобучения.
(Если `.onnx` не приехал — допишите его из `.pt`: `python manage.py export_onnx_models`.)

In [ ]:
from google.colab import files

models_dir = stem.parent
!cd {repo_root}/iot_hub/apps/telemetry/ml && zip -r /content/ml_models.zip models
files.download("/content/ml_models.zip")